# H-Neurons small Qwen experiment

This notebook is configured for Colab/Kaggle with a single T4 GPU. It uses a Hugging Face model ID directly, so you do not need to download the model manually first.

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [4]:
import os

In [1]:
import os
import pathlib
import subprocess

os.chdir('/content')

# Remove existing directory if it exists to force a re-clone
if pathlib.Path('H-Neuron-Implementation').exists():
    print('Removing existing H-Neuron-Implementation directory...')
    subprocess.run(['rm', '-rf', 'H-Neuron-Implementation'], check=True)

# Clone the repository
subprocess.run(['git', 'clone', '--depth=1', '-b', 'vkb', '--single-branch', 'https://github.com/CallmeAndree/H-Neuron-Implementation.git'], check=True)

os.chdir('H-Neuron-Implementation')
print('Working directory:', os.getcwd())

Removing existing H-Neuron-Implementation directory...
Working directory: /content/H-Neuron-Implementation


In [ ]:
import subprocess
import sys

# Colab/Kaggle setup without vLLM. Keep the default Colab PyTorch stack to avoid CUDA/vLLM kernel issues on T4.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers', 'accelerate', 'datasets', 'openai', 'scikit-learn', 'joblib', 'tqdm'
], check=True)

print('Installed dependencies without vLLM.')

In [5]:
# Use a Hugging Face model ID directly. Good for a T4 smoke test.
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

# Set your OpenAI-compatible key only if you run extract_answer_tokens.py.
# In Colab: from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# In Kaggle: use Add-ons > Secrets, then read it with kaggle_secrets.
# OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY')
# BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')

OUTPUT_DIR = 'data/small_subset'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('MODEL_ID =', MODEL_ID)

MODEL_ID = Qwen/Qwen2.5-1.5B-Instruct


## Collect Qwen responses

This generates Qwen-specific responses and rule-based correctness labels. For training, use 5 responses per question as requested. Increase `--max_samples` if you do not get enough balanced true/false samples.

In [ ]:
# vLLM is intentionally disabled in this notebook.
# Do NOT uninstall/reinstall torch or install vLLM here; Colab T4 often hits vLLM/FlashInfer CUDA kernel errors.
print('Skipping vLLM and CUDA reinstall steps.')

In [6]:
from google.colab import userdata
baby_key = userdata.get('baby_key')
phucbill_key = userdata.get('phucbill_key')
vkb_work_key = userdata.get('vkb_work_key')
biha_key = userdata.get('biha_key')
api_key = userdata.get('api_key')

In [7]:
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
BASE_URL_2 = "https://api.vietapi.tech/v1"
LLM_MODEL="models/gemini-3.1-flash-lite"
model = "gpt-5.4"

In [ ]:
import os

# Make accidental vLLM imports use safer settings, but this notebook no longer calls vLLM.
os.environ['VLLM_USE_V1'] = '0'
os.environ['VLLM_ATTENTION_BACKEND'] = 'TORCH_SDPA'
print('vLLM disabled for response collection; using Hugging Face Transformers instead.')

In [ ]:
import json
import os
import re
import string
import time
from pathlib import Path

import torch
from datasets import load_dataset
from openai import OpenAI
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def handle_punc(text):
        exclude = set(string.punctuation + '‘’´`')
        return ''.join(ch if ch not in exclude else ' ' for ch in text)
    if not s:
        return ''
    return white_space_fix(remove_articles(handle_punc(str(s).lower().replace('_', ' ')))).strip()


def load_existing_qids(path):
    if not os.path.exists(path):
        return set()
    qids = set()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                qids.update(json.loads(line).keys())
            except Exception:
                pass
    return qids


def rule_judge(response, norm_gts):
    norm_res = normalize_answer(response)
    return 'true' if any(gt and gt in norm_res for gt in norm_gts) else 'false'


judge_client = OpenAI(api_key=api_key, base_url=BASE_URL_2)

def llm_judge(question, response, answer_list):
    prompt = (
        f'Question: {question}\n'
        f'Response: {response}\n'
        f'Correct Answers: {answer_list}\n'
        "Please judge whether the response is correct or not. "
        "Return 't' if the response is correct, and 'f' if the response is incorrect. "
        "Don't add any additional information."
    )
    for attempt in range(5):
        try:
            completion = judge_client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
            )
            res = completion.choices[0].message.content.strip().lower()
            if 't' in res:
                return 'true'
            if 'f' in res:
                return 'false'
            print(f'Invalid judge response: {res}; retrying')
        except Exception as e:
            print(f'Judge API failed, attempt {attempt + 1}/5: {e}')
            time.sleep(2)
    return 'error'


DATA_PATH = '/content/H-Neuron-Implementation/data/TriviaQA/rc.nocontext/train-00000-of-00001.parquet'
OUTPUT_PATH = '/content/H-Neuron-Implementation/data/small_subset/test_qwen_samples.jsonl'
SAMPLE_NUM = 5
MAX_SAMPLES = 200
MAX_NEW_TOKENS = 50

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

print('Loading tokenizer/model with Transformers, not vLLM...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_lm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model_lm.eval()

terminators = []
if tokenizer.eos_token_id is not None:
    terminators.append(tokenizer.eos_token_id)
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
if isinstance(im_end_id, int) and im_end_id >= 0:
    terminators.append(im_end_id)
terminators = list(dict.fromkeys(terminators)) or None

dataset = load_dataset('parquet', data_files=DATA_PATH, split='train')
if MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))
processed_qids = load_existing_qids(OUTPUT_PATH)

all_correct_count = 0
all_incorrect_count = 0

with open(OUTPUT_PATH, 'a', encoding='utf-8') as f:
    for item in tqdm(dataset, desc='Sampling with Transformers + LLM judge'):
        qid = str(item.get('question_id', ''))
        if qid in processed_qids:
            continue

        question = item.get('question', '')
        if not question or 'answer' not in item:
            continue

        raw_aliases = []
        for col in ['aliases', 'normalized_aliases']:
            val = item['answer'].get(col)
            if val:
                raw_aliases.extend(val if isinstance(val, list) else [str(val)])
        norm_gts = [normalize_answer(a) for a in set(raw_aliases) if a]
        if not norm_gts:
            continue

        suffix = 'Respond with the answer only, without any explanation.'
        prompt_messages = [{'role': 'user', 'content': f'{question.strip()} {suffix}'}]
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors='pt').to(model_lm.device)
        prompt_len = inputs.input_ids.shape[-1]

        responses = []
        judges = []
        judge_cache = {}

        for _ in range(SAMPLE_NUM):
            with torch.inference_mode():
                output_ids = model_lm.generate(
                    **inputs,
                    do_sample=True,
                    temperature=1.0,
                    top_p=0.9,
                    top_k=50,
                    max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=terminators,
                    pad_token_id=tokenizer.eos_token_id,
                )
            ans = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            responses.append(ans)

            uncertain_terms = ["don't know", 'cannot', 'not provided', 'no information']
            if any(term in ans.lower() for term in uncertain_terms):
                judges.append('uncertain')
                continue

            if ans not in judge_cache:
                judge_cache[ans] = llm_judge(question, ans, raw_aliases)
            judges.append(judge_cache[ans])

        true_count = judges.count('true')
        if true_count == SAMPLE_NUM:
            all_correct_count += 1
        elif true_count == 0:
            all_incorrect_count += 1

        result = {
            qid: {
                'question': f'{question.strip()} {suffix}',
                'responses': responses,
                'judges': judges,
                'ground_truth': list(set(raw_aliases)),
            }
        }
        f.write(json.dumps(result, ensure_ascii=False) + '\n')
        f.flush()
        processed_qids.add(qid)

        if len(processed_qids) % 10 == 0:
            tqdm.write(f'Stats -> All-Correct: {all_correct_count}, All-Incorrect: {all_incorrect_count}')

print('Saved responses to', OUTPUT_PATH)

In [ ]:
from google.colab import userdata
baby_key = userdata.get('baby_key')
phucbill_key = userdata.get('phucbill_key')
vkb_work_key = userdata.get('vkb_work_key')
biha_key = userdata.get('biha_key')

In [ ]:
import os

output_file_path = '/content/H-Neuron-Implementation/data/small_subset/test_qwen_samples.jsonl'

if os.path.exists(output_file_path):
    print(f"File '{output_file_path}' exists.")
    file_size = os.path.getsize(output_file_path)
    print(f"File size: {file_size} bytes.")

    if file_size > 0:
        print("File content (first 5 lines):")
        with open(output_file_path, 'r') as f:
            for i, line in enumerate(f):
                print(line.strip())
                if i >= 4:  # Display first 5 lines
                    break
    else:
        print("File is empty.")
else:
    print(f"File '{output_file_path}' does not exist.")

File '/content/H-Neuron-Implementation/data/small_subset/test_qwen_samples.jsonl' exists.
File size: 0 bytes.
File is empty.


In [ ]:
import os

input_data_path = '/content/H-Neuron-Implementation/data/TriviaQA/rc.nocontext/test-00000-of-00001.parquet'

if os.path.exists(input_data_path):
    print(f"Input data file '{input_data_path}' exists. File size: {os.path.getsize(input_data_path)} bytes.")
else:
    print(f"Input data file '{input_data_path}' does not exist. This is likely why the output was empty.")
    print("You may need to download or generate the TriviaQA data.")

Input data file '/content/H-Neuron-Implementation/data/TriviaQA/rc.nocontext/test-00000-of-00001.parquet' exists. File size: 1199699 bytes.


In [ ]:
!python /content/H-Neuron-Implementation/h_neuron_scripts/extract_answer_tokens.py --input_path /content/H-Neuron-Implementation/data/small_subset/test_qwen_samples.jsonl --output_path /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl --tokenizer_path "Qwen/Qwen2.5-1.5B-Instruct" --api_keys {baby_key} {phucbill_key} {vkb_work_key} --base_url {BASE_URL} --llm_model {LLM_MODEL}  --rpm_limit 15 --rpd_limit 500 --resume

Using API key 1/3; today=0/500, minute=0/15
Resume enabled: found 0 already processed IDs in /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 0it [00:00, ?it/s]Saved tc_448 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 2it [00:03,  1.35s/it]Saved tc_964 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 3it [00:03,  1.08s/it]Saved tc_1071 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 4it [00:12,  4.24s/it]Saved tc_902 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 5it [00:13,  3.06s/it]Saved tc_434 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Processing tokens: 6it [00:14,  2.25s/it]Saved tc_19 to /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl
Pr

In [ ]:
!python /content/H-Neuron-Implementation/h_neuron_scripts/extract_activations.py \
  --model_path Qwen/Qwen2.5-1.5B-Instruct \
  --input_path /content/H-Neuron-Implementation/data/small_subset/test_answer_tokens_qwen.jsonl \
  --train_ids_path /content/H-Neuron-Implementation/data/small_subset/test_qwen_ids.json \
  --output_root data/small_subset/test_activations \
  --locations answer_tokens input output all_except_answer_tokens \
  --method mean


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 327.23it/s]
Loaded 169 target IDs for extraction.
Processing: 100% 169/169 [00:19<00:00,  8.50it/s]


In [ ]:
import shutil
import os
from google.colab import files

output_zip_name = 'test_activations.zip'
folder_to_zip = '/content/H-Neuron-Implementation/data/small_subset/test_activations'

# Create a zip archive of the folder
shutil.make_archive(output_zip_name.replace('.zip', ''), 'zip', folder_to_zip)

print(f"'{folder_to_zip}' has been zipped to '{output_zip_name}'.")
print("Downloading the zip file...")

# Download the zip file
files.download(output_zip_name)

'/content/H-Neuron-Implementation/data/small_subset/test_activations' has been zipped to 'test_activations.zip'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Download the zip file
files.download(output_zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
import os

source_path = '/content/H-Neuron-Implementation/test_activations.zip'
destination_path = '/content/drive/MyDrive/test_activations.zip'

if os.path.exists(source_path):
    shutil.move(source_path, destination_path)
    print(f"'{source_path}' moved to '{destination_path}'.")
else:
    print(f"File '{source_path}' does not exist. Cannot move.")

'/content/H-Neuron-Implementation/test_activations.zip' moved to '/content/drive/MyDrive/test_activations.zip'.
